# 🌐 Assistente Diversa — Educação Inclusiva
## Protótipo funcional para TCC

Este notebook entrega o protótipo funcional do TCC: um chat de Educação Inclusiva que usa uma base de conhecimento do Portal Diversa, busca por relevância com TF-IDF e geração de resposta com Llama 3.1 via Groq.

**Importante:** se você não informar a chave da Groq, o protótipo ainda funciona em modo demonstração, respondendo com base nos trechos recuperados da base.

## 1. Instalação de dependências



In [5]:
!pip install -q beautifulsoup4 gradio python-dotenv

print("✅ Instalação concluída!")
print()
print("Bibliotecas que já vêm no sistema")

print("  • requests   — chamadas HTTP (coleta e API)")
print("  • pandas     — manipulação de dados")
print("  • re         — expressões regulares (limpeza de texto)")
print("  • sklearn    — TfidfVectorizer e cosine_similarity")


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Reshimming asdf python...
✅ Instalação concluída!

Bibliotecas que já vêm no sistema
  • requests   — chamadas HTTP (coleta e API)
  • pandas     — manipulação de dados
  • re         — expressões regulares (limpeza de texto)
  • sklearn    — TfidfVectorizer e cosine_similarity


## 2. Importações e configuração



In [6]:
import json
import os
import re
import time
from typing import Dict, List

import gradio as gr
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from dotenv import load_dotenv

load_dotenv()
GROQ_KEY = os.getenv("GROQ_KEY", "")

GROQ_MODEL = "llama-3.1-8b-instant"
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

PARAMS_GERACAO = {
    "model": GROQ_MODEL,
    "max_tokens": 600,
    "temperature": 0.3,
    "top_p": 0.9,
}

print("✅ Configuração carregada")
print("Modelo:", GROQ_MODEL)
print("Modo:", "Groq/Llama" if GROQ_KEY else "Demonstração sem API")

/home/joaotrovaum/.asdf/installs/python/3.11.9/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


✅ Configuração carregada
Modelo: llama-3.1-8b-instant
Modo: Groq/Llama


## 3. Base de conhecimento


In [ ]:
import pandas as pd

sheet_id = "1Ll0gKNxMs20rMAESmjk4Dk2_FAtugKgY5NUhwaod1rI"
url_planilha = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

# 1. Transforma a planilha em um Pandas DataFrame
artigos_brutos = pd.read_csv(url_planilha)
artigos_brutos = artigos_brutos.rename(columns={"t": "titulo"})

# 2. Visualizando dimensões do Dataframe
print("Linhas x Colunas =", artigos_brutos.shape)


# 3. Imprime as colunas (corrigido: sem os parênteses no final)
print("\nColunas presentes na base de dados:")
print(list(artigos_brutos.columns))
print()

# 4. Visualiza as 5 primeiras linhas de forma formatada no Jupyter
print("5 primeiras linhas")
display(artigos_brutos.head())

# 5. Visualização dos tipos dos objetos
print(f"Todos os objetos são de texto:")
display(artigos_brutos.dtypes)

Linhas x Colunas = (34, 3)

Colunas presentes na base de dados:
['titulo', 'url', 'texto']

5 primeiras linhas


,titulo,url,texto
0,TEA na educação infantil: inclusão e afetivida...,https://diversa.org.br/artigos/tea-educacao-in...,O artigo aborda a inclusão de crianças com Tra...
1,Ferramentas para o desenvolvimento escolar de ...,https://diversa.org.br/artigos/ferramentas-par...,O artigo apresenta estratégias que podem apoia...
2,O que é Transtorno do Déficit de Atenção e Hip...,https://diversa.org.br/artigos/o-que-e-transto...,O artigo explica o Transtorno do Déficit de At...
3,Como a escola pode lidar com os diagnósticos m...,https://diversa.org.br/artigos/como-a-escola-p...,O artigo propõe uma reflexão cuidadosa sobre d...
4,Psicomotricidade como recurso para a educação ...,https://diversa.org.br/artigos/psicomotricidad...,O artigo apresenta a psicomotricidade como um ...


Todos os objetos são de texto:


titulo    object
url       object
texto     object
dtype: object

## 4. Limpeza da base



In [ ]:
def limpar_texto(texto):
    texto = re.sub(r"\s+", " ", str(texto))refactor: add dataset of inclusive education articles and update assistant prototype to incorporate the new data source
    texto = re.sub(r"\[.*?\]", "", texto)
    texto = re.sub(r"http\S+", "", texto)
    return texto.strip()

df = pd.DataFrame(artigos_brutos)
df["texto_limpo"] = df["texto"].apply(limpar_texto)
df = df[df["texto_limpo"].str.len() > 120].reset_index(drop=True)
df["conteudo_busca"] = df["titulo"] + ". " + df["titulo"] + ". " + df["texto_limpo"]

print("✅ Limpeza concluída")
print("Total de artigos válidos:", len(df))
display(df[["titulo", "url"]])

✅ Limpeza concluída
Total de artigos válidos: 34


,titulo,url
0,TEA na educação infantil: inclusão e afetivida...,https://diversa.org.br/artigos/tea-educacao-in...
1,Ferramentas para o desenvolvimento escolar de ...,https://diversa.org.br/artigos/ferramentas-par...
2,O que é Transtorno do Déficit de Atenção e Hip...,https://diversa.org.br/artigos/o-que-e-transto...
3,Como a escola pode lidar com os diagnósticos m...,https://diversa.org.br/artigos/como-a-escola-p...
4,Psicomotricidade como recurso para a educação ...,https://diversa.org.br/artigos/psicomotricidad...
5,Braile: acessibilidade para estudantes com def...,https://diversa.org.br/noticias/braile-acessib...
6,Educação de surdos: os desafios da proposta bi...,https://diversa.org.br/artigos/educacao-de-sur...
7,Qual o papel do Tradutor/Intérprete de Libras ...,https://diversa.org.br/artigos/qual-o-papel-do...
8,Flexibilizações vs. adaptações curriculares: c...,https://diversa.org.br/artigos/flexibilizacoes...
9,Os desafios de ser uma mulher com deficiência ...,https://diversa.org.br/artigos/os-desafios-de-...


## 5. Vetorização TF-IDF e busca semântica


In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    sublinear_tf=True,
    strip_accents="unicode",
    lowercase=True,
)

matriz_tfidf = vectorizer.fit_transform(df["conteudo_busca"])

def buscar_artigos_relevantes(pergunta, top_k=3, score_minimo=0.015):
    vetor = vectorizer.transform([pergunta])
    scores = cosine_similarity(vetor, matriz_tfidf)[0]
    indices = scores.argsort()[::-1][:top_k]

    resultados = []
    for i in indices:
        score = float(scores[i])
        if score >= score_minimo:
            resultados.append({
                "titulo": df.iloc[i]["titulo"],
                "url": df.iloc[i]["url"],
                "trecho": df.iloc[i]["texto_limpo"][:900],
                "score": round(score, 4),
            })
    return resultados

print("✅ Vetorização concluída")
print("Matriz:", matriz_tfidf.shape)

✅ Vetorização concluída
Matriz: (34, 2417)


## 6. Teste da busca

In [ ]:
pergunta_teste = "O que é Atendimento Educacional Especializado?"
resultados = buscar_artigos_relevantes(pergunta_teste)

print("Pergunta:", pergunta_teste)
print("Artigos recuperados:")
for r in resultados:
        print(f"- {r['titulo']} | score={r['score']}")

Pergunta: O que é Atendimento Educacional Especializado?
Artigos recuperados:
- O que é Atendimento Educacional Especializado (AEE)? | score=0.3656
- O que é a Política Nacional de Educação Especial na Perspectiva da Educação Inclusiva? | score=0.1317
- Braile: acessibilidade para estudantes com deficiência visual | score=0.124


## 7. Prompt do assistente e controle de escopo


In [ ]:
SYSTEM_PROMPT = (
"Você é um assistente especialista em Educação Inclusiva do Portal Diversa (diversa.org.br)."
"Seu papel é apoiar professores, gestores escolares, familiares de pessoas com deficiência, estudantes, "
        "pesquisadores e profissionais de secretarias de educação, usando os TRECHOS fornecidos."
        "RESPONDA apenas temas de educação inclusiva: autismo (TEA), TDAH, deficiência visual, auditiva, intelectual "
        "e física, AEE, sala de recursos multifuncionais, tecnologia assistiva, acessibilidade, adaptações curriculares, "
        "DUA, legislação educacional, família, gestão escolar e conteúdos do Portal Diversa."
        "NÃO RESPONDA: diagnóstico médico ou psicológico, recomendação de medicamento/tratamento, opinião política "
        "partidária, decisão judicial individual, nem assuntos sem relação com educação inclusiva."
        "Nunca invente URLs, títulos, leis, números ou fontes. Cite apenas fontes presentes nos TRECHOS fornecidos. "
        "Se a base não tiver informação suficiente, diga isso claramente."
        "Responda sempre em português brasileiro, com linguagem clara, acessível, empática e prática."
    )

PALAVRAS_ESCOPO = [
        "educação", "inclus", "deficiência", "deficiente", "autismo", "tea", "tdah", "aee", "lbi", "bncc",
        "ldb", "aluno", "estudante", "escola", "professor", "gestor", "família", "sala de recursos",
        "tecnologia assistiva", "acessibilidade", "libras", "braille", "curricular", "pedagóg", "aprendizagem",
        "apoio escolar", "laudo", "diagnóstico", "barreira", "adaptação", "desenho universal", "dua"
    ]

def pergunta_dentro_escopo(pergunta):
        pergunta_normalizada = limpar_texto(pergunta).lower()
        return any(palavra in pergunta_normalizada for palavra in PALAVRAS_ESCOPO)

print("✅ Prompt configurado")

✅ Prompt configurado


## 8. Funções de resposta



In [ ]:
def chamar_groq(messages, tentativas=3):
    if not GROQ_KEY:
        return ""

    headers = {
        "Authorization": f"Bearer {GROQ_KEY}",
        "Content-Type": "application/json",
    }
    payload = {**PARAMS_GERACAO, "messages": messages}

    for tentativa in range(1, tentativas + 1):
        try:
            resp = requests.post(GROQ_URL, headers=headers, json=payload, timeout=30)
            if resp.status_code == 200:
                return resp.json()["choices"][0]["message"]["content"].strip()
            if resp.status_code == 429:
                time.sleep(5 * tentativa)
                continue
            if resp.status_code == 401:
                return "❌ Chave da Groq inválida. Verifique a variável GROQ_KEY."
            return f"❌ Erro HTTP {resp.status_code}: {resp.text[:250]}"
        except Exception as exc:
            return f"❌ Erro inesperado: {exc}"
    return "⏳ Serviço indisponível no momento."

def resposta_demo_sem_llm(pergunta, artigos):
    if not artigos:
        return (
            "Minha especialidade é Educação Inclusiva. A base atual ainda não possui trechos suficientes "
            "para responder essa pergunta com segurança. Recomendo ampliar a base com mais artigos do Portal Diversa."
        )
    principal = artigos[0]
    return (
        "**Resposta em modo demonstração, sem API da Groq configurada:**\n\n"
        f"Com base no artigo **{principal['titulo']}**, a orientação principal é considerar a educação inclusiva "
        "como um processo de remoção de barreiras para que todos os estudantes participem da vida escolar. "
        "A escola deve planejar estratégias pedagógicas acessíveis, articular o Atendimento Educacional Especializado "
        "quando necessário e envolver professores, gestão, família e comunidade.\n\n"
        "Trecho recuperado da base:\n"
        f"> {principal['trecho'][:550]}...\n\n"
        "Para respostas mais completas geradas por IA, configure a variável `GROQ_KEY`."
    )

def montar_fontes(artigos):
    if not artigos:
        return ""
    linhas = ["\n\n---", "📚 **Fontes recuperadas da base:**"]
    for artigo in artigos:
        linhas.append(f"- [{artigo['titulo']}]({artigo['url']}) — score {artigo['score']}")
    return "\n".join(linhas)

def responder(pergunta, historico=None):
    pergunta = (pergunta or "").strip()
    if not pergunta:
        return "Digite uma pergunta sobre Educação Inclusiva."

    artigos = buscar_artigos_relevantes(pergunta, top_k=3)

    if not pergunta_dentro_escopo(pergunta) and not artigos:
        return "Minha especialidade é Educação Inclusiva. Posso te ajudar com alguma dúvida sobre esse tema?"

    contexto = "\n\n".join(
        f"Artigo: {a['titulo']}\nURL: {a['url']}\nTrecho: {a['trecho']}\nScore: {a['score']}"
        for a in artigos
    ) or "Nenhum trecho relevante foi encontrado na base atual."

    instrucao_usuario = (
        "Use os TRECHOS DO PORTAL DIVERSA abaixo para responder. "
        "Não invente fontes. Caso a informação não esteja nos trechos, diga que a base atual é insuficiente.\n\n"
        f"TRECHOS DO PORTAL DIVERSA:\n{contexto}\n\n"
        f"PERGUNTA DO USUÁRIO: {pergunta}"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instrucao_usuario},
    ]

    resposta = chamar_groq(messages)
    if not resposta:
        resposta = resposta_demo_sem_llm(pergunta, artigos)

    return resposta + montar_fontes(artigos)

print("✅ Funções criadas")

✅ Funções criadas


## 9. Testes automáticos do protótipo




In [ ]:
testes = [
    ("Dentro do escopo", "O que é Atendimento Educacional Especializado (AEE)?"),
    ("Dentro do escopo", "Como adaptar atividades para alunos com autismo?"),
    ("Dentro do escopo", "Como a tecnologia assistiva ajuda estudantes com deficiência visual?"),
    ("Fora do escopo", "Qual é a capital da França?"),
    ("Caso-limite", "Como ajudar uma criança?"),
]

for categoria, pergunta in testes:
    print("=" * 80)
    print("Categoria:", categoria)
    print("Pergunta:", pergunta)
    print("-" * 80)
    print(responder(pergunta)[:1200])
    print()

Categoria: Dentro do escopo
Pergunta: O que é Atendimento Educacional Especializado (AEE)?
--------------------------------------------------------------------------------
**Resposta em modo demonstração, sem API da Groq configurada:**

Com base no artigo **O que é Atendimento Educacional Especializado (AEE)?**, a orientação principal é considerar a educação inclusiva como um processo de remoção de barreiras para que todos os estudantes participem da vida escolar. A escola deve planejar estratégias pedagógicas acessíveis, articular o Atendimento Educacional Especializado quando necessário e envolver professores, gestão, família e comunidade.

Trecho recuperado da base:
> O Atendimento Educacional Especializado, conhecido como AEE, é um serviço da educação especial que identifica, elabora e organiza recursos pedagógicos e de acessibilidade para eliminar barreiras à participação dos estudantes. Ele deve complementar ou suplementar a escolarização, nunca substituir a sala de aula comum. O

## 10. Interface de chat com Gradio


In [3]:
import gradio as gr
import base64

CAMINHO_CORDAO = "./imagens_do_tcc/CordaoTEA.svg"

with open(CAMINHO_CORDAO, "rb") as arquivo:
    cordao_base64 = base64.b64encode(arquivo.read()).decode("utf-8")

CORDÃO_TEA_URL = f"data:image/svg+xml;base64,{cordao_base64}"

CAMINHO_ICONE_GRUPO = "./imagens_do_tcc/images.svg"

with open(CAMINHO_ICONE_GRUPO, "rb") as arquivo:
    icone_grupo_base64 = base64.b64encode(arquivo.read()).decode("utf-8")

ICONE_GRUPO_URL = f"data:image/svg+xml;base64,{icone_grupo_base64}"

css_personalizado = f"""
body {{
    background: #FAFAFA;
}}

.gradio-container {{
    max-width: 1200px !important;
    margin: auto !important;
    font-family: 'Inter', sans-serif !important;
    background: #FAFAFA !important;
}}

/* Caixa superior */
#hero {{
    background: #FFFFFF;
    border-radius: 26px;
    padding: 34px;
    color: #111827;
    box-shadow: 0 8px 24px rgba(0, 0, 0, 0.05);
    margin-bottom: 24px;
    position: relative;
    overflow: hidden;
}}

/* Cordão como borda decorativa do topo */
#hero::before {{
    content: "";
    position: absolute;
    inset: 0;
    border-radius: 26px;
    padding: 10px;
    background-image: url("{CORDÃO_TEA_URL}");
    background-size: 180px;
    background-repeat: repeat;
    background-position: center;
    opacity: 0.95;

    -webkit-mask:
        linear-gradient(#fff 0 0) content-box,
        linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;

    pointer-events: none;
}}

#hero h1 {{
    font-size: 32px;
    margin-bottom: 8px;
    color: #111827;
}}

#hero p {{
    font-size: 16px;
    line-height: 1.6;
    color: #374151;
}}

/* Caixa do chat */
#chatbot {{
    border-radius: 24px !important;
    background: #FFFFFF !important;
    box-shadow: 0 8px 24px rgba(0,0,0,0.05);
    position: relative;
    overflow: hidden;
}}

/* Borda do cordão no chat */
#chatbot::before {{
    content: "";
    position: absolute;
    inset: 0;
    border-radius: 24px;
    padding: 9px;
    background-image: url("{CORDÃO_TEA_URL}");
    background-size: 160px;
    background-repeat: repeat;
    background-position: center;
    opacity: 0.95;

    -webkit-mask:
        linear-gradient(#fff 0 0) content-box,
        linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;

    pointer-events: none;
    z-index: 2;
}}

/* Cards laterais */
.card-info {{
    background: #FFFFFF;
    border-radius: 22px;
    padding: 22px;
    box-shadow: 0 6px 18px rgba(0, 0, 0, 0.05);
    position: relative;
    overflow: hidden;
}}

.card-info::before {{
    content: "";
    position: absolute;
    inset: 0;
    border-radius: 22px;
    padding: 8px;
    background-image: url("{CORDÃO_TEA_URL}");
    background-size: 140px;
    background-repeat: repeat;
    background-position: center;
    opacity: 0.95;

    -webkit-mask:
        linear-gradient(#fff 0 0) content-box,
        linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;

    pointer-events: none;
}}

.card-info h3 {{
    color: #111827;
    margin-top: 0;
}}

.small-text {{
    font-size: 14px;
    color: #374151;
    line-height: 1.6;
}}

/* Campo de pergunta */
textarea {{
    border-radius: 14px !important;
    border: 1px solid #D1D5DB !important;
    background: #FFFFFF !important;
}}

/* Botões */
button {{
    border-radius: 14px !important;
    font-weight: 600 !important;
}}

/* Botão enviar */
#btn-enviar {{
    background: #111827 !important;
    color: white !important;
    border: none !important;
}}

#btn-enviar:hover {{
    background: #000000 !important;
}}

/* Botão limpar */
#btn-limpar {{
    background: #FFFFFF !important;
    color: #111827 !important;
    border: 1px solid #D1D5DB !important;
}}

/* Perguntas sugeridas */
.pergunta-btn button {{
    background: #FFFFFF !important;
    color: #111827 !important;
    border: 1px solid #D1D5DB !important;
    box-shadow: 0 3px 10px rgba(0,0,0,0.06);
}}

.pergunta-btn button:hover {{
    background: #F3F4F6 !important;
}}

.footer {{
    text-align: center;
    color: #6B7280;
    font-size: 13px;
    margin-top: 18px;
}}

.icone-grupo {{
    width: 44px;
    height: 44px;
    object-fit: contain;
    margin-right: 12px;
}}

.titulo-grupo {{
    display: flex;
    align-items: center;
    gap: 10px;
    margin: 0;
    color: #111827;
    font-size: 32px;
}}
"""
# ============================================================
# FUNÇÃO DO CHAT
# Pergunta
# ============================================================

def conversar(pergunta, historico):
    if historico is None:
        historico = []

    if pergunta is None or pergunta.strip() == "":
        return "", historico

    resposta = responder(pergunta)

    historico.append((pergunta, resposta))

    return "", historico


def limpar_chat():
    return []


# ============================================================
# TEMA DO GRADIO
# ============================================================

tema = gr.themes.Soft(
    primary_hue="green",
    neutral_hue="slate",
    font=gr.themes.GoogleFont("Inter")
)

# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(
    theme=tema,
    css=css_personalizado,
    title="Assistente Diversa — Educação Inclusiva"
) as demo:

    gr.HTML(f"""
<div id="hero">
    <h1 class="titulo-grupo">
        <img src="{ICONE_GRUPO_URL}" class="icone-grupo">
        <span>Equipe Hélice</span>
    </h1>

    <p>
        Protótipo funcional para TCC 
    </p>
</div>
""")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Conversa com o Assistente",
                elem_id="chatbot",
                height=520
            )

            pergunta = gr.Textbox(
                label="Digite sua pergunta",
                placeholder="Exemplo: O que é AEE? Como incluir um aluno com TEA?",
                lines=2,
                elem_id="pergunta-textbox" # Added elem_id for specific styling
            )

            with gr.Row():
                btn_enviar = gr.Button("Enviar pergunta", elem_id="btn-enviar")
                btn_limpar = gr.Button("Limpar conversa", elem_id="btn-limpar")

        with gr.Column(scale=1):
            gr.HTML("""
            <div class="card-info">
                <h3>📚 Sobre o protótipo</h3>
                <p class="small-text">
                    Este assistente responde com base em uma base de conhecimento
                    cadastrada manualmente com fontes reais sobre educação inclusiva.
                </p>
                <p class="small-text">
                    Ele não substitui professores, profissionais do AEE, médicos,
                    psicólogos ou especialistas.
                </p>
            </div>
            """)

            gr.HTML("<br>")

            gr.HTML("""
            <div class="card-info">
                <h3>✅ Perguntas sugeridas</h3>
                <p class="small-text">Clique em uma pergunta para preencher o campo de texto.</p>
            </div>
            """)

            btn_p1 = gr.Button("O que é educação inclusiva?", elem_classes="pergunta-btn")
            btn_p2 = gr.Button("O que é AEE?", elem_classes="pergunta-btn")
            btn_p3 = gr.Button("Como incluir aluno com TEA?", elem_classes="pergunta-btn")
            btn_p4 = gr.Button("O que diz a LBI sobre inclusão?", elem_classes="pergunta-btn")
            btn_p5 = gr.Button("O que são tecnologias assistivas?", elem_classes="pergunta-btn")

    gr.HTML("""
    <div class="footer">
        Projeto acadêmico desenvolvido para fins educacionais — SoulCode / Accenture
    </div>
    """)

    # Eventos dos botões principais
    btn_enviar.click(
        fn=conversar,
        inputs=[pergunta, chatbot],
        outputs=[pergunta, chatbot]
    )

    pergunta.submit(
        fn=conversar,
        inputs=[pergunta, chatbot],
        outputs=[pergunta, chatbot]
    )

    btn_limpar.click(
        fn=limpar_chat,
        inputs=None,
        outputs=chatbot
    )

    # Eventos das perguntas sugeridas
    btn_p1.click(lambda: "O que é educação inclusiva?", outputs=pergunta)
    btn_p2.click(lambda: "O que é Atendimento Educacional Especializado?", outputs=pergunta)
    btn_p3.click(lambda: "Como incluir um aluno com TEA em sala de aula?", outputs=pergunta)
    btn_p4.click(lambda: "O que a Lei Brasileira de Inclusão fala sobre educação?", outputs=pergunta)
    btn_p5.click(lambda: "O que são tecnologias assistivas na educação inclusiva?", outputs=pergunta)


demo.launch(share=True)

C:\Users\Cesar\AppData\Local\Temp\ipykernel_7484\4237269681.py:253: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://d2f0dbd19735d2e87f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
